# Day 071 — Exercise 4: search_by_text + search_by_image

**What you'll build:** The two query functions for Vision RAG — text search and reverse image search.

**Why it matters:** These are the two user-facing interfaces. A product search feature might expose both: type a query or upload a photo to find similar items.

In [ ]:
import base64, hashlib, io
import numpy as np
from PIL import Image

_SEARCH_PROMPT = (
    'Describe this image in detail for use in a semantic search index. '
    'Include: main subjects, colors, textures, setting, and visible text. '
    'Write one concise paragraph of 2-3 sentences.'
)

def image_to_base64(img, format='PNG'):
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()

def cosine_similarity(a, b):
    va = np.array(a, dtype=np.float32)
    vb = np.array(b, dtype=np.float32)
    denom = float(np.linalg.norm(va) * np.linalg.norm(vb))
    if denom == 0.0:
        return 0.0
    return float(np.dot(va, vb) / denom)

class ImageIndex:
    def __init__(self):
        self._items = []
    def add(self, image_id, description, embedding, metadata=None):
        self._items.append({'id': image_id, 'description': description,
                            'embedding': np.array(embedding, dtype=np.float32),
                            'metadata': metadata or {}})
    def search(self, query_embedding, n=5):
        if not self._items:
            return []
        q = np.array(query_embedding, dtype=np.float32)
        scored = [(cosine_similarity(q, item['embedding']), item) for item in self._items]
        scored.sort(key=lambda x: -x[0])
        top = scored[:min(n, len(scored))]
        return [{'id': item['id'], 'description': item['description'],
                 'score': float(score), 'metadata': item['metadata']}
                for score, item in top]
    def __len__(self):
        return len(self._items)

def describe_image_for_search(img, describe_fn=None):
    img_b64 = image_to_base64(img)
    if describe_fn is not None:
        return describe_fn(img_b64, _SEARCH_PROMPT)
    import ollama
    resp = ollama.chat(model='llava',
                       messages=[{'role': 'user', 'content': _SEARCH_PROMPT, 'images': [img_b64]}])
    return resp['message']['content'].strip()

def embed_text(text, embed_fn=None):
    if embed_fn is not None:
        return embed_fn(text)
    import ollama
    resp = ollama.embeddings(model='nomic-embed-text', prompt=text)
    return resp['embedding']

def index_images(images_with_ids, describe_fn=None, embed_fn=None):
    index = ImageIndex()
    for image_id, img, metadata in images_with_ids:
        desc = describe_image_for_search(img, describe_fn=describe_fn)
        emb  = embed_text(desc, embed_fn=embed_fn)
        index.add(image_id, desc, emb, metadata or {})
    return index

def search_by_text(query, index, embed_fn=None, n=5):
    q_emb = embed_text(query, embed_fn=embed_fn)
    return index.search(q_emb, n=n)

def search_by_image(img, index, describe_fn=None, embed_fn=None, n=5):
    desc  = describe_image_for_search(img, describe_fn=describe_fn)
    q_emb = embed_text(desc, embed_fn=embed_fn)
    return index.search(q_emb, n=n)

import hashlib
def _mock_describe(img_b64, prompt):
    h = int(hashlib.md5(img_b64.encode()).hexdigest()[:4], 16)
    labels = ['a red apple on a table', 'a blue ocean wave',
              'a green forest path', 'a yellow sunflower field']
    return labels[h % len(labels)]

def _mock_embed(text):
    h = int(hashlib.md5(text.encode()).hexdigest()[:8], 16)
    return [((h >> (i * 8)) & 0xff) / 128.0 - 1.0 for i in range(4)]

# Pre-built index for checks
_imgs = [
    ('img_r', Image.new('RGB', (16,16), (220, 50, 50)), {'tag': 'red'}),
    ('img_b', Image.new('RGB', (16,16), (50, 100, 220)), {'tag': 'blue'}),
    ('img_g', Image.new('RGB', (16,16), (50, 180, 80)), {'tag': 'green'}),
]
_index = index_images(_imgs, describe_fn=_mock_describe, embed_fn=_mock_embed)


## Task

**`search_by_text(query, index, embed_fn=None, n=5) -> list[dict]`:**
- `q_emb = embed_text(query, embed_fn=embed_fn)`
- `return index.search(q_emb, n=n)`

**`search_by_image(img, index, describe_fn=None, embed_fn=None, n=5) -> list[dict]`:**
- `desc = describe_image_for_search(img, describe_fn=describe_fn)`
- `q_emb = embed_text(desc, embed_fn=embed_fn)`
- `return index.search(q_emb, n=n)`

## Your Implementation

In [ ]:
def search_by_text(query: str, index: ImageIndex,
                   embed_fn=None, n: int = 5) -> list:
    """Search the index by a text query.

    Embeds the query, then returns top-n results from index.search.
    """
    raise NotImplementedError


def search_by_image(img, index: ImageIndex,
                    describe_fn=None, embed_fn=None, n: int = 5) -> list:
    """Search the index using an image as the query.

    Describes the query image, embeds the description, returns top-n results.
    """
    raise NotImplementedError


In [ ]:
def search_by_text(query, index, embed_fn=None, n=5):
    q_emb = embed_text(query, embed_fn=embed_fn)
    return index.search(q_emb, n=n)


def search_by_image(img, index, describe_fn=None, embed_fn=None, n=5):
    desc  = describe_image_for_search(img, describe_fn=describe_fn)
    q_emb = embed_text(desc, embed_fn=embed_fn)
    return index.search(q_emb, n=n)


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # search_by_text returns a list
    results = search_by_text('a red object', _index, embed_fn=_mock_embed, n=3)
    assert isinstance(results, list)
    score += 1; print("\u2705 search_by_text returns a list")

    # n=3 returns at most 3 results
    assert len(results) <= 3
    score += 1; print("\u2705 search_by_text respects n limit")

    # each result has required keys
    assert all('id' in r and 'score' in r and 'description' in r and 'metadata' in r
               for r in results)
    score += 1; print("\u2705 search_by_text results have all required keys")

    # search_by_image returns a list with correct keys
    query_img = Image.new('RGB', (16, 16), (200, 80, 80))
    img_results = search_by_image(query_img, _index,
                                  describe_fn=_mock_describe,
                                  embed_fn=_mock_embed, n=2)
    assert isinstance(img_results, list) and len(img_results) <= 2
    assert all('id' in r and 'score' in r for r in img_results)
    score += 1; print("\u2705 search_by_image returns correct result list")

    # results sorted descending by score
    if len(results) >= 2:
        assert results[0]['score'] >= results[1]['score']
    score += 1; print("\u2705 results sorted by score descending")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def search_by_text(query, index, embed_fn=None, n=5):
    q_emb = embed_text(query, embed_fn=embed_fn)
    return index.search(q_emb, n=n)


def search_by_image(img, index, describe_fn=None, embed_fn=None, n=5):
    desc  = describe_image_for_search(img, describe_fn=describe_fn)
    q_emb = embed_text(desc, embed_fn=embed_fn)
    return index.search(q_emb, n=n)
```

**Why are both functions so short?** All the logic is in the components they compose. Short functions that do one thing and compose other tested functions are easier to debug than long functions with mixed concerns.

</details>